In [1]:
import os
os.getcwd()


'/mnt/lustre/proj/eunbyeol/Hackathon2025_AG/diurnal_cycle'

In [2]:
from dask.distributed import Client
client = Client(scheduler_file='/proj/eunbyeol/MPI/scheduler.json')
client

<Client: 'tcp://203.247.189.225:41776' processes=7 threads=126, memory=586.73 GiB>

In [3]:
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import dask

dask.config.set({"array.slicing.split_large_chunks": False})

In [6]:
# User settings

exp_dir = Path("/proj/shared_data/awicm3")
# exp = "TCo1279-DART-1950C"
# ys = 1950
# ye = 1969

exp = "TCo1279-DART-2080C"
ys = 2080
ye = 2092


indata_dir = "outdata/oifs"
analysis_dir = "analysis/oifs/clim_3h"

sdir = Path("/proj/eunbyeol/Hackathon2025_AG/data_clim/")

vars = ["10ws", "10u", "10v", "2t", "lsp", "cp"]
freq = "3h"
prefix = "atm_remapped"




In [ ]:

# for var in vars[:1]:

for var in vars:    
    print(f"\n===== Processing variable {var} =====")

    indir = exp_dir / exp / indata_dir / freq / var
    outdir = sdir / exp / analysis_dir / var
    outdir.mkdir(parents=True, exist_ok=True)

    for month in range(1, 13):
    # for month in range(1, 2):
        mm = f"{month:02d}"
        print(f"  Processing month {mm}")

        files = []
        for year in range(ys, ye + 1):
            fname = (
                f"{prefix}_{freq}_{var}_{freq}_"
                f"{year}{mm}-{year}{mm}.nc"
            )
            fpath = indir / fname
            if fpath.exists():
                files.append(str(fpath))
            else:
                print(f"    WARNING: missing {fpath}")

        # print(f"    Files used for {var}, month {mm}:")
        # for f in files:
        #     print(f"      {f}")

        if not files:
            print(f"    No files for {var}, month {mm}")
            continue

        ds = xr.open_mfdataset(
            files,
            combine="by_coords",
            parallel=False,
            chunks={"time_counter": -1},
            # engine="netcdf4"
        )
        #-- add2 -------------------------
        ds = ds.assign_coords(
            time_counter=ds.time_counter - pd.Timedelta(minutes=1)
        )
        ds = ds.sel(
            time_counter=ds.time_counter.dt.month == int(mm)
        )

        cnt = ds.groupby("time_counter.hour").count("time_counter")
        # print(cnt)
        #-- add2 -------------------------

        # ----------------------------------
        # climatological diurnal cycle (yhourmean)
        # ----------------------------------
        ds_hour = (
            ds
            .groupby("time_counter.hour")
            .mean("time_counter", keep_attrs=True)
        )

        # hour → time_counter
        hours = ds_hour["hour"].values
        time_counter = pd.to_datetime(
            [f"2000-01-01 {int(h):02d}:00:00" for h in hours]
        )

        ds_hour = (
            ds_hour
            .assign_coords(time_counter=("hour", time_counter))
            .swap_dims({"hour": "time_counter"})
            .drop_vars("hour")
        )

        outname = (
            f"{prefix}_{freq}_{var}_{ys}-{ye}."
            f"{mm}.diurnal_cycle.nc"
        )
        outpath = outdir / outname

        print(f"    Saving: {outpath}")

        ds_hour.compute().to_netcdf(outpath)
        ds.close()

print("All variables done.")


===== Processing variable 10ws =====
  Processing month 01
    Saving: /proj/eunbyeol/Hackathon2025_AG/data_clim/TCo1279-DART-2080C/analysis/oifs/clim_3h/10ws/atm_remapped_3h_10ws_2080-2092.01.diurnal_cycle.nc
  Processing month 02
    Saving: /proj/eunbyeol/Hackathon2025_AG/data_clim/TCo1279-DART-2080C/analysis/oifs/clim_3h/10ws/atm_remapped_3h_10ws_2080-2092.02.diurnal_cycle.nc
  Processing month 03
    Saving: /proj/eunbyeol/Hackathon2025_AG/data_clim/TCo1279-DART-2080C/analysis/oifs/clim_3h/10ws/atm_remapped_3h_10ws_2080-2092.03.diurnal_cycle.nc
  Processing month 04
    Saving: /proj/eunbyeol/Hackathon2025_AG/data_clim/TCo1279-DART-2080C/analysis/oifs/clim_3h/10ws/atm_remapped_3h_10ws_2080-2092.04.diurnal_cycle.nc
  Processing month 05
    Saving: /proj/eunbyeol/Hackathon2025_AG/data_clim/TCo1279-DART-2080C/analysis/oifs/clim_3h/10ws/atm_remapped_3h_10ws_2080-2092.05.diurnal_cycle.nc
  Processing month 06
    Saving: /proj/eunbyeol/Hackathon2025_AG/data_clim/TCo1279-DART-2080C/an

## Figure

In [ ]:
fname = (
    "/proj/eunbyeol/Hackathon2025_AG/data_clim/TCo1279-DART-1950C/"
    "analysis/oifs/clim_3h/"
    "atm_remapped_3h_10u_1950-1969.01.yhourmean.nc"
)

ds = xr.open_dataset(fname)
print(ds)


In [ ]:
fname2 = (
    "/proj/eunbyeol/Hackathon2025_AG/data_clim/alex_results/"
    "atm_reduced_3h_10u_3h_1950-1969.yhourmean.H00.ymonmean.remapcon.nc"
)

ds2 = xr.open_dataset(fname2)
print(ds2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ds["10u"].isel(time_counter=0).plot(
    ax=axes[0],
    cmap="viridis",
    add_colorbar=True
)
axes[0].set_title("2m Temperature (00 UTC)")

ds2["10u"].isel(time_counter=0).plot(
    ax=axes[1],
    cmap="viridis",
    add_colorbar=True
)
axes[1].set_title("2m Temperature (00 UTC) - CDO")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ds["2t"].isel(time=0).plot(
    ax=axes[0],
    cmap="viridis",
    add_colorbar=True
)
axes[0].set_title("10ws at time index 0 (00 UTC)")

ds2["2t"].isel(time_counter=0).plot(
    ax=axes[1],
    cmap="viridis",
    add_colorbar=True
)
axes[1].set_title("10ws at time index 0 (00 UTC) - CDO")

plt.tight_layout()
plt.show()


In [ ]:

da1 = ds["10u"].isel(time_counter=0)
da2 = ds2["10u"].isel(time_counter=0)
diff = da1 - da2


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ds
da1.plot(
    ax=axes[0],
    cmap="viridis",
    add_colorbar=True
)
axes[0].set_title("Python (time=0)")

# ds2 (CDO)
da2.plot(
    ax=axes[1],
    cmap="viridis",
    add_colorbar=True
)
axes[1].set_title("CDO (time=0)")

# difference
diff.plot(
    ax=axes[2],
    cmap="RdBu_r",
    add_colorbar=True,
    center=0
)
axes[2].set_title("Python - CDO")

plt.tight_layout()
plt.show()
